[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ironyr/llm-project/blob/feat/v1/notebooks/02_lora_finetuning.ipynb)

**Google Colab:** Set `GIT_REPO` to your fork, run the bootstrap cell, then `ensure_knowledge_base`. **Use Runtime → Change runtime type → GPU (T4 or better)** for 4-bit LoRA.

**Saving weights:** By default the adapter is saved under `./models/lora_nust_bank` in the runtime (ephemeral). Mount Google Drive in a cell and set `finetune.output_dir` in `config.yaml` or override `out_dir` in the training section to a path under `/content/drive/MyDrive/...`.

In [ ]:
# === Colab / local bootstrap (run first) ===
import os
import subprocess
import sys
from pathlib import Path

# Colab: set to your fork’s HTTPS clone URL (must match the repo that hosts this notebook)
GIT_REPO = "https://github.com/IronYR/llm-project.git"
GIT_BRANCH = "feat/v1"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.chdir("/content")
    root = Path("/content/llm-project")
    if not (root / "config.yaml").is_file():
        if "YOUR_GITHUB_USERNAME" in GIT_REPO:
            raise RuntimeError(
                "Edit GIT_REPO to your GitHub fork, e.g. https://github.com/<you>/llm-project.git"
            )
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_REPO, str(root)],
            check=True,
        )
    os.chdir(root)
else:
    p = Path.cwd().resolve()
    if p.name == "notebooks":
        os.chdir(p.parent)
    elif not (p / "config.yaml").is_file() and (p.parent / "config.yaml").is_file():
        os.chdir(p.parent)

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

if IN_COLAB:
    from src.colab_setup import colab_pip_install
    colab_pip_install(ROOT)

print("ROOT =", ROOT)


In [ ]:
from src.colab_setup import ensure_knowledge_base

ensure_knowledge_base(ROOT)

# LoRA fine-tuning (PEFT + TRL + bitsandbytes)

Interactive version of `train_finetune.py`, in the style of [nust-bank-chatbot/notebooks/FineTuning.ipynb](https://github.com/geetu040/nust-bank-chatbot/blob/main/notebooks/FineTuning.ipynb).

**This repo uses:** causal LM **Qwen2.5-3B-Instruct**, **LoRA** via `peft`, **SFTTrainer** from `trl`, optional **4-bit** loads with `bitsandbytes` on CUDA — not Flan-T5 / Seq2Seq.

**Before running:** `python ingest.py` and install deps (`pip install -r requirements.txt`). **GPU with CUDA** strongly recommended for 4-bit training.

## Load config and training data

In [ ]:
from src.finetune_lib import (
    attach_lora,
    build_dataset_rows,
    build_sft_dataset,
    build_sft_training_args,
    load_base_model_for_training,
    load_config,
    load_tokenizer,
)

cfg = load_config(ROOT / "config.yaml")
ft = cfg.get("finetune", {})
data_cfg = cfg.get("data", {})
model_id = ft.get("base_model_id", "Qwen/Qwen2.5-3B-Instruct")
out_dir = Path(ft.get("output_dir", "./models/lora_nust_bank"))
out_dir.mkdir(parents=True, exist_ok=True)

rows = build_dataset_rows(data_cfg["processed_dir"], data_cfg["knowledge_base_file"])
print(f"Training examples: {len(rows)}")
assert rows, "No rows — run: python ingest.py"

tokenizer = load_tokenizer(model_id)
train_ds = build_sft_dataset(tokenizer, rows)
train_ds

## Load base model and attach LoRA

On **CUDA**, `use_4bit: true` in `config.yaml` loads the model in 4-bit. Otherwise the model loads in fp16 on CPU (or MPS if `use_mps: true`).

In [ ]:
model = load_base_model_for_training(model_id, ft)
model = attach_lora(model, ft)
model.print_trainable_parameters()

## Train with SFTTrainer

In [ ]:
from trl import SFTTrainer

sft_cfg = build_sft_training_args(ft, out_dir)

try:
    trainer = SFTTrainer(
        model=model,
        args=sft_cfg,
        train_dataset=train_ds,
        processing_class=tokenizer,
    )
except TypeError:
    trainer = SFTTrainer(
        model=model,
        args=sft_cfg,
        train_dataset=train_ds,
        tokenizer=tokenizer,
    )

# First step can take a long time on CPU
trainer.train()

## Save adapter and tokenizer

In [ ]:
trainer.save_model(str(out_dir))
tokenizer.save_pretrained(str(out_dir))
print("Saved to", out_dir.resolve())